# **Fragility of saliency maps and defenses: attack, input shift, β-smoothing, certificate**

Practice for the module [“Attacks on Explanations”](https://ai-interpretability.school/learn/22), lessons “Fragility of saliency maps”, “Geometry and model manipulation” and “Defenses and certification”.

In the lessons a saliency map broke in four ways, and defenses were proposed against two of them. Here all of this is computed on one small convolutional network for MNIST that we train in a couple of minutes:

1. **the attack of Ghorbani, Abid, and Zou** ([AAAI 2019](https://arxiv.org/abs/1710.10547)): an input perturbation in the ball $\|\delta\|_\infty\le\varepsilon$ moves the map away from the original one while the predicted class stays the same; the point of comparison is a random perturbation of the same budget;
2. **the input invariance test of Kindermans et al.** ([2019](https://arxiv.org/abs/1711.00867)): the input is shifted by a constant, the bias of the first layer compensates the shift, and we check which `captum.attr` methods respond to a shift the model does not respond to;
3. **the targeted manipulation of Dombrowski et al.** ([NeurIPS 2019](https://arxiv.org/abs/1906.07983)): the map of one image is turned into the map of another image while the network outputs stay almost the same;
4. **β-smoothing** from the same paper: the map is computed on a network where ReLU is replaced by $\mathrm{softplus}_\beta$;
5. **softplus and weight decay during training** ([Dombrowski et al., 2022](https://arxiv.org/abs/2012.10425)): the network is trained from the start with a smooth activation and a weight penalty;
6. **Sparsified SmoothGrad of Levine, Singla, and Feizi** ([2019](https://arxiv.org/abs/1905.12105)): a smoothed sparse map and a certificate — a lower bound on how many top-$K$ pixels stay in the top-$K$ under any perturbation of radius $\rho$.

We measure map robustness with our own functions (top-$k$ intersection, Spearman and Pearson correlations) and with two metrics of the Quantus library: `quantus.MaxSensitivity` and `quantus.LocalLipschitzEstimate`.

All randomness is fixed with seed 0. Network training and attack iterations depend on hardware and library versions, so your numbers may differ in the last digits; the conclusions do not change.

In [ ]:
!pip install -q captum==0.9.0 quantus==0.6.0

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # Captum and Quantus warn about hooks and input format; tqdm about ipywidgets
import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import quantus
from scipy.stats import spearmanr
from torch.distributions import Normal
from torchvision import datasets, transforms
from captum.attr import Saliency, InputXGradient, IntegratedGradients, NoiseTunnel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
np.random.seed(0)
print('device:', device)

## 1. Data and network

MNIST is loaded from `torchvision` without keys. Pixels are in $[0,1]$, with no normalization: in section 3 we shift the input by a constant ourselves.

The network is small: two convolutions, average pooling and two fully connected layers. Two choices in it are made for the checks below:

- **the first convolution has no padding.** Padding writes zeros at the border, and a zero after the input shift no longer corresponds to a zero before the shift, so the compensation of Kindermans et al. stops being exact at the borders;
- **the pooling is average, not max.** On the black MNIST background max pooling sees many equal values, and which one wins is decided by rounding in the last digit. Then even the gradients of two mathematically identical networks diverge, and the check starts measuring rounding instead of the method.

The function `make_net(act)` takes the activation class: this way the same architecture is built with ReLU and with softplus.

In [ ]:
to_tensor = transforms.ToTensor()
train_set = datasets.MNIST('data', train=True, download=True, transform=to_tensor)
test_set = datasets.MNIST('data', train=False, download=True, transform=to_tensor)
X_test = test_set.data[:2000].float().div(255).unsqueeze(1).to(device)
Y_test = test_set.targets[:2000].to(device)


def make_net(act):
    return nn.Sequential(nn.Conv2d(1, 16, 5), act(), nn.Conv2d(16, 32, 5), act(), nn.AvgPool2d(2),
                         nn.Flatten(), nn.Linear(32 * 10 * 10, 64), act(), nn.Linear(64, 10)).to(device)


def train(net, weight_decay=0.0, epochs=2, seed=0):
    torch.manual_seed(seed)
    loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,
                                         generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=weight_decay)
    net.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            F.cross_entropy(net(xb.to(device)), yb.to(device)).backward()
            opt.step()
    return net.eval()


def accuracy(net):
    with torch.no_grad():
        return (net(X_test).argmax(1) == Y_test).float().mean().item()


t0 = time.time()
torch.manual_seed(0)
net = train(make_net(nn.ReLU))
print(f'ReLU network: accuracy on 2000 test images {accuracy(net):.4f}, training {time.time() - t0:.0f} s')

Common functions of the notebook:

- `saliency_map(model, x, c, create_graph)` — the plain gradient map, as in Ghorbani et al.: the absolute gradient of the score of class $c$ with respect to the input, normalized so that the sum over pixels is one. With `create_graph=True` the map stays differentiable with respect to the input, and an attack can follow its gradient;
- `smooth_copy(model, beta)` — a copy of the network with the same weights where every `nn.ReLU` is replaced by `nn.Softplus(beta)`;
- `topk_overlap(a, b, k)` — the share of common pixels among the $k$ most important pixels of two maps;
- `spearman(a, b)` and `pearson(a, b)` — correlations of two maps for every row of the batch.

We attack and check the first 50 test images that the network classifies correctly.

In [ ]:
def saliency_map(model, x, c, create_graph=False):
    x = x if x.requires_grad else x.clone().requires_grad_(True)
    score = model(x)[torch.arange(len(x)), c].sum()
    g, = torch.autograd.grad(score, x, create_graph=create_graph)
    a = g.abs().flatten(1)
    return a / a.sum(1, keepdim=True)


def smooth_copy(model, beta):
    m = copy.deepcopy(model)
    for i, layer in enumerate(m):
        if isinstance(layer, nn.ReLU):
            m[i] = nn.Softplus(beta)
    return m.eval()


def topk_overlap(a, b, k):
    ia = a.topk(k, 1).indices
    ib = b.topk(k, 1).indices
    hit = torch.zeros_like(a, dtype=torch.bool).scatter_(1, ia, True).gather(1, ib)
    return hit.float().sum(1) / k


def spearman(a, b):
    a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    return np.array([spearmanr(p, q).statistic for p, q in zip(a, b)])


def pearson(a, b):
    a = a - a.mean(1, keepdim=True)
    b = b - b.mean(1, keepdim=True)
    return (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1))


with torch.no_grad():
    pred = net(X_test).argmax(1)
correct = torch.nonzero(pred == Y_test).flatten()
x0, c0 = X_test[correct[:50]], pred[correct[:50]]
K_TOP = 78                                  # 10 % of 784 pixels
h0 = saliency_map(net, x0, c0)
print('batch:', tuple(x0.shape), 'classes:', c0[:10].tolist(), '...')

## 2. The attack of Ghorbani, Abid, and Zou

### 2.1. Why the attack needs softplus

The top-k attack maximizes $D(x)=-\sum_{i\in B}I(x)_i$: the attack lowers the total importance that the map $I(x)$ assigns to the pixels $B$, the most important ones on the original map. An attack step follows the sign of $\nabla_x D$, and $D$ itself is built from a gradient. So the second derivative of the network with respect to the input is needed.

**Task 1. Compute $\nabla_x D$ for the first image of the batch twice: through the ReLU network `net` and through its copy `smooth_copy(net, 5.0)`. The set $B$ is the `K_TOP` most important pixels of the original map of the ReLU network. How many of the 784 gradient elements are nonzero in each case?**

In [ ]:
# Your code here

For the ReLU network the gradient of $D$ is identically zero, and this is not rounding. Inside every region where the signs of the ReLU inputs do not change, the network is a linear function of the input: convolutions, average pooling and fully connected layers are linear, and ReLU on such a region is multiplication by a constant mask. The gradient of a linear function does not depend on the input, its derivative with respect to the input is zero. That is why Ghorbani et al., and Dombrowski et al. after them, run the attack through a softplus copy of the network and check the finished perturbation on the original ReLU network.

### 2.2. Top-k attack

**Task 2. Write the function `ghorbani_topk(model, x, c, eps, beta=5.0, steps=300)`. It makes `steps` steps $x\leftarrow\Pi\big(x+\alpha\,\mathrm{sign}\,\nabla_x D\big)$ with $\alpha=\varepsilon/10$, where the gradient of $D$ is computed through `smooth_copy(model, beta)` and $\Pi$ is the projection onto the ball $\|x-x_t\|_\infty\le\varepsilon$ and onto the pixel range $[0,1]$. After every step $D$ is recomputed on the original `model`; for every image the function returns the perturbed image with the largest $D$ on the original network among those with the original class.**

The number of steps is $P=300$, as in Algorithm 1 of the paper. The authors take the step $\alpha=0.5$ on the 0–255 pixel scale with budgets $\varepsilon$ up to 8; on MNIST in the $[0,1]$ scale the budgets are larger, so here the step is a fraction of the budget.

Hint: if the class changed at every step, return the original image — the attack simply failed for it.

In [ ]:
# Your code here

Let us compare the attack with the point of comparison of Ghorbani et al. — a random perturbation of the same budget: every pixel is shifted by $\pm\varepsilon$ with a random sign. For every $\varepsilon$ we print the share of images with the original class, the mean top-78 intersection of the original and the new map and the mean Spearman correlation between them.

In [ ]:
rows, attacked = [], {}
for eps in (0.05, 0.1, 0.2):
    gen = torch.Generator().manual_seed(0)
    x_rand = (x0 + eps * (torch.randint(0, 2, x0.shape, generator=gen).to(device) * 2 - 1)).clamp(0, 1)
    attacked[eps] = ghorbani_topk(net, x0, c0, eps)
    for kind, xx in (('random ±ε', x_rand), ('top-k attack', attacked[eps])):
        h1 = saliency_map(net, xx, c0)
        with torch.no_grad():
            same = (net(xx).argmax(1) == c0).float().mean().item()
        rows.append({'ε': eps, 'perturbation': kind, 'class kept': same,
                     'top-78 intersection': topk_overlap(h0, h1, K_TOP).mean().item(),
                     'Spearman': spearman(h0, h1).mean()})
table_attack = pd.DataFrame(rows)
print(table_attack.round(3).to_string(index=False))

In [ ]:
i = 0
xa = attacked[0.1][i:i + 1]
panels = ((x0[i, 0], 'original image'), (xa[0, 0], 'after the attack, ε = 0.1'),
          (h0[i].view(28, 28), 'map of the original'), (saliency_map(net, xa, c0[i:i + 1])[0].view(28, 28), 'map after the attack'))
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img.detach().cpu(), cmap='gray' if 'image' in title or 'attack,' in title else 'hot')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.suptitle(f'Class {c0[i].item()} before and after the attack; top-78 intersection of the maps — '
             f'{topk_overlap(h0[i:i + 1], saliency_map(net, xa, c0[i:i + 1]), K_TOP).item():.2f}', fontsize=11)
fig.tight_layout()
plt.show()

### 2.3. A hyperparameter of the attack: $\beta$ of the softplus copy

$\beta$ decides how close the softplus copy is to the ReLU network. With a small $\beta$ the copy is smooth, but its gradient describes the real network poorly; with a large $\beta$ the copy almost coincides with the ReLU network, while its second derivative is concentrated near the kinks and is close to zero almost everywhere. Let us check how the success of the attack at $\varepsilon=0.1$ depends on $\beta$.

In [ ]:
for beta in (1.0, 5.0, 20.0, 100.0):
    xb = ghorbani_topk(net, x0, c0, 0.1, beta=beta)
    print(f'β = {beta:>5}: top-78 intersection after the attack {topk_overlap(h0, saliency_map(net, xb, c0), K_TOP).mean():.3f}')

## 3. Shifting the input by a constant: the test of Kindermans et al.

The network `net` sees pixels in $[0,1]$. Let us build a second network `net_shift` that sees the same images in the encoding $[-1,0]$, i.e. $x_2=x_1+m_2$ with $m_2=-1$, and outputs the same numbers on corresponding inputs. It is enough to fix the bias of the first convolution: $b_2=b_1-\sum W\,m_2$, where the sum runs over the input channel and the kernel window. All other weights are shared.

An explanation method passes the test if the maps of the two networks on corresponding inputs coincide. We measure the relative discrepancy $\|a_1-a_2\|_2/\|a_1\|_2$ and take the largest one over the batch. The threshold 0.01 has a margin on both sides: the discrepancy of the methods that pass is of the order of float32 rounding error, of those that fail — of the order of one.

In [ ]:
M2 = -1.0
net_shift = copy.deepcopy(net)
with torch.no_grad():
    net_shift[0].bias -= M2 * net_shift[0].weight.sum((1, 2, 3))
x1, x2 = x0, x0 + M2
with torch.no_grad():
    print(f'largest difference between the outputs of the two networks: {(net(x1) - net_shift(x2)).abs().max().item():.1e}')


def rel_diff(a1, a2):
    return ((a1 - a2).flatten(1).norm(dim=1) / a1.flatten(1).norm(dim=1)).max().item()

**Task 3. Compute `rel_diff` of the maps of the two networks for five `captum.attr` methods and put them into a table “method — discrepancy — passes the test (discrepancy < 0.01)”. Methods: `Saliency` (with `abs=False`), `InputXGradient` (Gradient×Input), `IntegratedGradients` with a zero baseline for both networks, `IntegratedGradients` with the “black image” baseline (0 for `net`, $m_2$ for `net_shift`), `NoiseTunnel(Saliency)` in SmoothGrad mode (16 copies, `stdevs=0.15`). Use `n_steps=64` for IG. The SmoothGrad noise is random: set `torch.manual_seed(0)` before the call for each network so that both networks get the same noise. How many of the five methods pass the test?**

In [ ]:
# Your code here

Why Gradient×Input fails is visible without the table: the gradients of the two networks are the same, but they are multiplied by different inputs, so the difference of the attributions equals $\nabla_x f\odot m_2$. Let us check this identity and look at the maps.

In [ ]:
a1, a2 = pairs['Gradient×Input']
grad = Saliency(net).attribute(x1, target=c0, abs=False)
print(f'largest difference between a2 - a1 and grad * m2: {(a2 - a1 - grad * M2).abs().max().item():.1e}')

i = 0
names = ('Saliency', 'Gradient×Input', 'IG, zero reference', 'IG, black image')
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, name in enumerate(names):
    for row, (a, title) in enumerate(zip(pairs[name], ('pixels in [0, 1]', 'pixels in [-1, 0]'))):
        m = a[i, 0].detach().cpu()
        lim = m.abs().max()
        axes[row, col].imshow(m, cmap='bwr', vmin=-lim, vmax=lim)
        axes[row, col].set_title(f'{name}\n{title}', fontsize=9)
        axes[row, col].axis('off')
fig.tight_layout()
plt.show()

### 3.1. Where the test itself breaks: padding

The compensation through the bias is exact only if every output of the first layer sees real pixels only. Let us build the same first convolution with padding 2 and compare the outputs of the two encodings: inside the image they coincide, while at the border the padding writes a zero that in the encoding $[-1,0]$ already means a gray pixel, not a black one.

In [ ]:
conv_pad = nn.Conv2d(1, 16, 5, padding=2).to(device)
conv_pad.load_state_dict(net[0].state_dict())
conv_pad_shift = copy.deepcopy(conv_pad)
with torch.no_grad():
    conv_pad_shift.bias -= M2 * conv_pad_shift.weight.sum((1, 2, 3))
    diff = (conv_pad(x1) - conv_pad_shift(x2)).abs().amax((0, 1))
print(f'largest output difference inside (rows and columns 2..25): {diff[2:-2, 2:-2].max().item():.1e}')
print(f'largest output difference at the border: {diff.max().item():.2f}')

## 4. Targeted manipulation: Dombrowski et al.

Now the attack has a target: the map $h^t$ of another image. The manipulated image $x_{adv}$ is found by minimizing

$$\mathcal{L}=\frac{\|h(x_{adv})-h^t\|^2}{\|h^t\|^2}+\gamma\,\big\|p(x_{adv})-p(x)\big\|^2,$$

where $p$ are the class probabilities of the ReLU network. The first term pulls the map toward the target (we divide it by $\|h^t\|^2$ so that it is of order one), the second keeps the outputs in place. The authors use network outputs in the second term and no norm budget; we keep the probabilities and add the budget $\|x_{adv}-x\|_\infty\le\varepsilon$, otherwise on MNIST the optimization simply redraws the digit.

Pairs: the first 20 images of the batch are the originals, the next 20 are the sources of target maps; the target $h^t$ is the map of the second image of a pair for its own class.

**Task 4. Write the function `dombrowski(expl_model, attack_model, x, c, h_t, eps, gamma=10.0, steps=300, lr=0.01)`. It minimizes $\mathcal{L}$ over $x_{adv}$ with the Adam optimizer, computing the map $h$ through `attack_model` with `create_graph=True` and the probabilities through `net`; after every step $x_{adv}$ is projected onto the ball $\|x_{adv}-x\|_\infty\le\varepsilon$ and onto $[0,1]$. The parameter `expl_model` tells the attack whose map it will finally fool; in this section `attack_model = smooth_copy(net, 5.0)` and `expl_model = net`.**

In [ ]:
# Your code here

We print the share of pairs with the original class, the Pearson correlation of the map with the target before and after the attack, the correlation of the new map with the original one and the median change of probabilities $\|p(x_{adv})-p(x)\|_1$.

In [ ]:
xs, cs = x0[:20], c0[:20]
xt, ct = x0[20:40], c0[20:40]


def manipulation_row(expl_model, attack_model, eps, label):
    h_t = saliency_map(expl_model, xt, ct)
    h_s = saliency_map(expl_model, xs, cs)
    xa = dombrowski(expl_model, attack_model, xs, cs, h_t, eps)
    h_a = saliency_map(expl_model, xa, cs)
    with torch.no_grad():
        same = (net(xa).argmax(1) == cs).float().mean().item()
        dp = (F.softmax(net(xa), 1) - F.softmax(net(xs), 1)).abs().sum(1).median().item()
    return xa, {'map': label, 'ε': eps, 'class kept': same,
                'Pearson with target before': pearson(h_s, h_t).mean().item(),
                'Pearson with target after': pearson(h_a, h_t).mean().item(),
                'Pearson with original': pearson(h_a, h_s).mean().item(), 'median Δp': dp}


x_manip, row = manipulation_row(net, smooth_copy(net, 5.0), 0.2, 'ReLU')
print(pd.DataFrame([row]).round(3).to_string(index=False))

In [ ]:
i = 0
h_t = saliency_map(net, xt[i:i + 1], ct[i:i + 1])
panels = ((xs[i, 0], 'original', 'gray'), (x_manip[i, 0], 'manipulated', 'gray'), (xt[i, 0], 'target source', 'gray'),
          (saliency_map(net, xs[i:i + 1], cs[i:i + 1])[0].view(28, 28), 'map of the original', 'hot'),
          (saliency_map(net, x_manip[i:i + 1], cs[i:i + 1])[0].view(28, 28), 'map of the manipulated', 'hot'),
          (h_t[0].view(28, 28), 'target map', 'hot'))
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img.detach().cpu(), cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
with torch.no_grad():
    fig.suptitle(f'Class of the original {cs[i].item()}, of the manipulated {net(x_manip[i:i + 1]).argmax().item()}, '
                 f'of the target source {ct[i].item()}', fontsize=11)
fig.tight_layout()
plt.show()

## 5. β-smoothing

By theorem 1 of Dombrowski et al., the change of the map when the input moves along the surface of constant output is bounded by a quantity proportional to $\beta$. The defense is to compute the map on `smooth_copy(net, β)` with a small $\beta$. Let us test it against an attack that knows about the defense: the attack is now aimed directly at the smoothed map, i.e. `expl_model = attack_model = smooth_copy(net, β)`, and the attack gradient is exact.

In [ ]:
rows = [row]
for beta in (10.0, 3.0, 1.0):
    smoothed = smooth_copy(net, beta)
    rows.append(manipulation_row(smoothed, smoothed, 0.2, f'β = {beta}')[1])
table_beta = pd.DataFrame(rows)
print(table_beta.round(3).to_string(index=False))

β-smoothing has a limitation: `smooth_copy(net, β)` is a different function, and its map explains a network that is not quite the one that makes the prediction. Let us see, for every $\beta$, how often the predictions of the copy agree with those of the network and how similar the map of the copy is to the map of the ReLU network.

In [ ]:
with torch.no_grad():
    pred_relu = net(X_test).argmax(1)
for beta in (100.0, 10.0, 3.0, 1.0):
    smoothed = smooth_copy(net, beta)
    with torch.no_grad():
        agree = (smoothed(X_test).argmax(1) == pred_relu).float().mean().item()
    sim = pearson(saliency_map(smoothed, x0, c0), h0).mean().item()
    print(f'β = {beta:>5}: prediction agrees with the ReLU network on {agree:.3f} of images, Pearson of the map with the ReLU map {sim:.3f}')

## 6. Softplus and weight decay during training: Dombrowski et al., 2022

β-smoothing changes only the way the map is computed. The second paper of Dombrowski et al. changes the model itself: the bound on the norm of the Hessian of the output with respect to the input decreases together with the weight norms and with $\Sigma_2$, the bound on the second derivative of the activation. Hence two of the three ways: train the network with softplus from the start and with weight decay. The third way, a penalty on the Hessian norm, requires third-order derivatives at every training step, and we do not repeat it here.

Let us train the same architecture with `nn.Softplus(10)` and weight decay $5\cdot10^{-4}$ — the values of $\beta$ and $\lambda$ from table 1 of the paper.

In [ ]:
t0 = time.time()
torch.manual_seed(0)
net_soft = train(make_net(lambda: nn.Softplus(10.0)), weight_decay=5e-4)
print(f'network with softplus β=10 and weight decay: accuracy {accuracy(net_soft):.4f}, training {time.time() - t0:.0f} s')

The robustness measure of the paper is the Pearson correlation between the map of an input and the map of the same input with Gaussian noise of level $\nu$. Let us compare three models: the ReLU network, its β-smoothed copy with $\beta=3$ and the network trained with softplus and weight decay. The noise is the same for all models.

In [ ]:
models = {'ReLU': net, 'β-smoothing β=3': smooth_copy(net, 3.0), 'trained with softplus and weight decay': net_soft}
noise = torch.randn(x0.shape, generator=torch.Generator().manual_seed(0)).to(device)
rows = []
for nu in (0.05, 0.1, 0.2):
    row = {'ν': nu}
    for name, m in models.items():
        with torch.no_grad():
            c = m(x0).argmax(1)
        row[name] = pearson(saliency_map(m, x0, c), saliency_map(m, x0 + nu * noise, c)).mean().item()
    rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 6.1. The same models in Quantus metrics

`quantus.MaxSensitivity` ([Yeh et al., 2019](https://arxiv.org/abs/1901.09392)) takes `nr_samples` random input perturbations in a ball of radius `lower_bound` and returns the largest change of the map divided by the Frobenius norm of the original map (the `norm_denominator` parameter); `quantus.LocalLipschitzEstimate` ([Alvarez-Melis and Jaakkola, 2018](https://arxiv.org/abs/1806.08049)) — the largest ratio of the change of the map to the change of the input under Gaussian noise with standard deviation `perturb_std`. The measure $\mathrm{SENS}_{\max}$ of Yeh et al. is not divided by the norm of the map, while Quantus divides by default, so numbers under the two definitions cannot be compared. For both metrics lower is more robust. Quantus gets the explanation from a function `explain_func(model, inputs, targets)` that takes and returns NumPy arrays.

**Task 5. Compute both metrics for the three models of the dictionary `models` on the batch `x0`: `nr_samples=10`, `lower_bound=0.05` for `MaxSensitivity`, `perturb_std=0.05` for `LocalLipschitzEstimate`, the explanation is Captum `Saliency`, the class is the prediction of the same model. Before every metric call set `np.random.seed(0)`: Quantus draws noise from NumPy. Print the table “model — accuracy — mean MaxSensitivity — mean LocalLipschitzEstimate”.**

In [ ]:
# Your code here

## 7. Sparsified SmoothGrad: a certificate instead of a heuristic

Levine, Singla, and Feizi prove for any map $h$ with values in $[0,1]$ and its smoothing $\bar h(x)=\mathbb{E}_{\epsilon\sim\mathcal N(0,\sigma^2I)}\,h(x+\epsilon)$: if

$$\Phi\Big(\Phi^{-1}\big(\bar h_{[i]}(x)\big)-\frac{2\rho}{\sigma}\Big)\ \ge\ \bar h_{[2K-i]}(x),$$

then under any perturbation $\|\tilde x-x\|_2\le\rho$ at least $i$ pixels of the top-$K$ of the smoothed map stay in the top-$K$. Here $\bar h_{[j]}$ is the $j$-th largest element and $\Phi$ is the cumulative distribution function of the standard normal law. The certificate $R_{cert}$ is the largest such $i$.

Two base maps:

- **SmoothGrad**: the absolute gradient of every noisy copy divided by its own maximum so that the values fall into $[0,1]$;
- **Sparsified SmoothGrad**: in every noisy copy the share $\tau$ of the largest absolute gradient elements gets 1, the rest get 0.

We estimate the smoothed map by the mean over $q$ copies. A strict certificate requires confidence bounds for every $\bar h_{[j]}$; first we compute the certificate from point estimates, and the strict variant comes at the end of the section. We take $K=196$, a quarter of the pixels, as in the authors' MNIST experiment.

**Task 6. Write two functions. `smooth_map(model, x, c, sigma, q, tau=None, seed=0)` returns a smoothed map of shape `(len(x), 784)`: with `tau=None` — SmoothGrad with every copy normalized by its maximum, otherwise Sparsified SmoothGrad with share `tau` (the number of ones is `int(tau * 784)`). Take the noise from `torch.Generator().manual_seed(seed)`; it is convenient to compute the copies in packs of 128. `certificate(h, K, rho, sigma)` returns for every row of `h` the largest $i\le K$ for which the condition of the theorem holds, or 0.**

In [ ]:
# Your code here

The certified share of the top-$K$, $R_{cert}/K$, averaged over 10 images, for three noise levels $\sigma$ and three radii $\rho$; $q=256$ copies, $\tau=0.1$.

In [ ]:
K_CERT = 196
xc, cc = x0[:10], c0[:10]
rows, maps = [], {}
for sigma in (0.25, 0.5, 1.0):
    maps[sigma] = (smooth_map(net, xc, cc, sigma, 256), smooth_map(net, xc, cc, sigma, 256, tau=0.1))
    for rho in (0.1, 0.25, 0.5):
        rows.append({'σ': sigma, 'ρ': rho,
                     'SmoothGrad': certificate(maps[sigma][0], K_CERT, rho, sigma).float().mean().item() / K_CERT,
                     'Sparsified SmoothGrad': certificate(maps[sigma][1], K_CERT, rho, sigma).float().mean().item() / K_CERT})
table_cert = pd.DataFrame(rows)
print(table_cert.round(3).to_string(index=False))

A certificate is a lower bound, and it can be compared with what happens in practice. Take $\sigma=0.5$, $\rho=0.25$ and, for each of the 10 images, 20 random perturbations with $\|\delta\|_2=\rho$; compute Sparsified SmoothGrad of the perturbed input with another noise seed and the smallest share of common top-$K$ pixels over the perturbations. A random perturbation is not the worst one, so the observed share must be no lower than the certificate, but it may be higher: the certificate only promises what survives any perturbation.

In [ ]:
sigma, rho = 0.5, 0.25
h_clean = maps[sigma][1]
cert = certificate(h_clean, K_CERT, rho, sigma).float() / K_CERT
gen = torch.Generator().manual_seed(1)
worst = torch.ones(len(xc), device=device)
for r in range(20):
    d = torch.randn(xc.shape, generator=gen).to(device)
    d = rho * d / d.flatten(1).norm(dim=1).view(-1, 1, 1, 1)
    h_pert = smooth_map(net, xc + d, cc, sigma, 256, tau=0.1, seed=100 + r)
    worst = torch.min(worst, topk_overlap(h_clean, h_pert, K_CERT))
print('certificate R_cert / K:       ', np.round(cert.numpy(), 2))
print('observed minimum over 20:     ', np.round(worst.cpu().numpy(), 2))

### 7.1. The strict variant: confidence bounds

Every $\bar h_j$ is the mean of $q$ values from $[0,1]$, and by Hoeffding's inequality, with probability at least $1-\alpha$ simultaneously for all 784 pixels, the estimate differs from the true value by at most $e=\sqrt{\ln(2\cdot784/\alpha)/(2q)}$. The strict certificate puts the lower bound $\bar h_{[i]}-e$ into the left-hand side of the condition and the upper bound $\bar h_{[2K-i]}+e$ into the right-hand side. Let us see how much of the certificate is left at $\alpha=0.05$ and different $q$.

In [ ]:
def certificate_strict(h, K, rho, sigma, q, alpha=0.05):
    e = np.sqrt(np.log(2 * h.shape[1] / alpha) / (2 * q))
    normal = Normal(0.0, 1.0)
    hs = h.sort(1, descending=True).values.double().cpu()
    i = torch.arange(1, K + 1)
    lhs = normal.cdf(normal.icdf((hs[:, :K] - e).clamp(1e-12, 1 - 1e-12)) - 2 * rho / sigma)
    ok = lhs >= hs[:, 2 * K - i - 1] + e
    return torch.where(ok.any(1), (ok * i).amax(1), torch.zeros(len(h), dtype=torch.long)), e


for q in (256, 1024, 4096):
    h_q = smooth_map(net, xc, cc, 0.5, q, tau=0.1)
    strict, e = certificate_strict(h_q, K_CERT, 0.25, 0.5, q)
    point = certificate(h_q, K_CERT, 0.25, 0.5)
    print(f'q = {q:>4}: e = {e:.3f}, certificate from point estimates {point.float().mean() / K_CERT:.3f}, '
          f'strict {strict.float().mean() / K_CERT:.3f}')

**Task 7. Using the tables of sections 5–7: which of the defenses gives a guarantee for a specific input and radius, and which only lower the sensitivity of the map empirically? What does each of them change — the way the map is computed or the model itself — and what is paid for it (accuracy, agreement with the explained network, number of passes)?**

**Your answer:**

## What to take away from the notebook

- **An unchanged prediction does not guarantee an unchanged map.** The top-k attack of Ghorbani et al. in an $L_\infty$ ball moves the map more than a random perturbation of the same budget, and the class changes for none of the images. The point of comparison is mandatory: part of the discrepancy comes from random noise alone.
- **For a ReLU network the second derivative with respect to the input is identically zero**, so the attack runs through a softplus copy and is checked on the original network. How well the attack works depends on the $\beta$ of that copy.
- **A constant input shift compensated in the bias does not change the model, but it changes Gradient×Input and IG with a zero baseline.** The plain gradient, SmoothGrad over it and IG with a baseline that shifts together with the data pass the test. The test itself requires that the first layer does not see padding.
- **The targeted manipulation of Dombrowski et al. pulls the map toward the map of another image** while the class probabilities stay almost the same.
- **β-smoothing and training with softplus and weight decay lower the sensitivity of the map but guarantee nothing.** β-smoothing also explains a different function: with a small $\beta$ the predictions of the copy diverge from those of the network.
- **Sparsified SmoothGrad gives a certificate**, and it is more informative than the certificate of plain SmoothGrad; it grows with $\sigma$ and decreases with $\rho$. A strict certificate requires confidence bounds, and with them thousands of noisy copies per input.